In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage
import operator

In [17]:
load_dotenv()

generator_llm = ChatCohere(model="command-r7b-12-2024")
evaluation_llm = ChatCohere(model="command-r7b-12-2024")
optimizer_llm = ChatCohere(model="command-r7b-12-2024")

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

structured_evaluation_llm = evaluation_llm.with_structured_output(TweetEvaluation)


In [23]:
class TweetState(TypedDict):
    topic : str
    tweet : str
    evaluation : Literal['approved','needs_improvement']
    feedback : str
    no_iterations : int
    max_iterations : int
    tweet_history : Annotated[list[str],operator.add]
    feedback_history : Annotated[list[str],operator.add]

def generate_tweet(state : TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    response = generator_llm.invoke(messages).content

    return {'tweet':response, 'tweet_history':[response]}

def evaluate_tweet(state : TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_evaluation_llm.invoke(messages)

    return {'evaluation':response.evaluation,'feedback':response.feedback ,'feedback_history':[response]}

def optimize_tweet(state : TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content
    itr = state['no_interations']+1;

    return {'tweet':response, 'no_iterations':itr, 'tweet_history':[response]}

def route_evaluation(state : TweetState):
    if(state['evaluation']=='approved' or no_iterations>=max_iterations):
        return 'approved'
    else:
        return 'needs_improvement'



In [27]:
graph = StateGraph(TweetState)

graph.add_node('generate_tweet',generate_tweet)
graph.add_node('evaluate_tweet',evaluate_tweet)
graph.add_node('optimize_tweet',optimize_tweet)

graph.add_edge(START,'generate_tweet')
graph.add_edge('generate_tweet','evaluate_tweet')
graph.add_conditional_edges('evaluate_tweet',route_evaluation,{'approved':END, 'needs_improvement':'optimize_tweet'})
graph.add_edge('optimize_tweet','evaluate_tweet')

workflow = graph.compile()

initial_state = {
    'topic':'srhberhb',
    'max_iterations':5,
    'no_iterations':1
                 }
workflow.invoke(initial_state)


{'topic': 'srhberhb',
 'tweet': 'SRHBerhb: When your boss says "Let\'s brainstorm," and you\'re like, "Brainstorm what? I\'m out of ideas and coffee." #OfficeStruggles',
 'evaluation': 'approved',
 'feedback': 'This tweet is a clever play on the common office struggle of feeling uninspired during brainstorming sessions. The use of the hashtag #OfficeStruggles adds a relatable context and a sense of community, making it more likely to resonate with a wide audience. The tweet is short, sharp, and scroll-stopping, with a punchy delivery that captures the frustration many employees feel in such situations. It also avoids the pitfalls of question-answer format, excessive length, and generic punchlines, making it a well-formed and humorous tweet with potential for virality.',
 'no_iterations': 1,
 'max_iterations': 5,
 'tweet_history': ['SRHBerhb: When your boss says "Let\'s brainstorm," and you\'re like, "Brainstorm what? I\'m out of ideas and coffee." #OfficeStruggles'],
 'feedback_history